# Step 3-4: FashionCLIP embeddings and the board "vibe vector"

This notebook loads a pretrained FashionCLIP model and uses it to turn your board images into a single numeric "vibe vector" -- an average of each image's embedding, representing the overall aesthetic of the board.

## Load the board images

This notebook is self-contained, so let's re-list the images in `data/board_images/` the same way notebook 01 did.

In [1]:
from pathlib import Path

board_dir = Path("../data/board_images")
board_image_paths = sorted([
    p for p in board_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
])

print(f"Found {len(board_image_paths)} board images")

Found 10 board images


## Load FashionCLIP

We're using **`patrickjohncyh/fashion-clip`**, a version of OpenAI's CLIP model fine-tuned on fashion product images and captions. Regular CLIP already learned a shared space for images and text from millions of general internet image/caption pairs; FashionCLIP adapts that space specifically for clothing, so it's better at picking up on garment details than the vanilla version.

CLIP itself is two models glued together:
- an **image encoder** that turns a picture into a vector of numbers
- a **text encoder** that turns a sentence/word into a vector of numbers, *in the same vector space*

That shared space is what makes later steps (zero-shot aesthetic tags, garment-detail extraction) possible — we can compare an image vector to a text vector directly with cosine similarity, even though one started as pixels and the other as words. Right now we only need the image encoder side.

The `transformers` library gives us two objects:
- a **model** (the actual neural network with its trained weights)
- a **processor** (handles preprocessing: resizing/cropping/normalizing images into the exact tensor format the model expects)

The first time this runs, it downloads the model weights (a few hundred MB) from Hugging Face and caches them locally — subsequent runs will be instant.

In [2]:
import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "patrickjohncyh/fashion-clip"

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

print("Model loaded.")

Using device: mps


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded.


## Sanity check: embed one image

Before looping over everything, let's embed a single image and look at the result, so we know exactly what shape of data we're working with.

One wrinkle worth knowing: in current `transformers`, `model.get_image_features(...)` doesn't hand back a bare tensor — it returns an output object, and the actual embedding lives on its `.pooler_output` attribute (shape `[batch, 512]`). We pull that out explicitly below.

In [3]:
from PIL import Image

image = Image.open(board_image_paths[0]).convert("RGB")

inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.get_image_features(**inputs)

image_embedding = outputs.pooler_output

print("Embedding shape:", image_embedding.shape)
print(image_embedding[0, :10])

Embedding shape: torch.Size([1, 512])


tensor([-0.2666,  2.2579,  0.6809,  0.2255, -0.0195,  0.3100, -0.9741,  0.0146,
         0.1364, -0.1640], device='mps:0')


## Embed every board image, then average into one "vibe vector"

Now the real step: repeat that same embedding process for every image in the board, then average all the resulting vectors together element-by-element. The idea is that each image is one data point describing part of the board's aesthetic, and averaging blends them into a single vector that summarizes the board as a whole -- the same way averaging a list of numbers gives you one number representing the group.

We also **L2-normalize** the final vector (scale it so its length is 1). This matters for later steps: cosine similarity, which we'll use for recommendations and zero-shot tagging, measures the *angle* between vectors, not their length -- normalizing here keeps things consistent with how we'll compare it against other embeddings.

In [4]:
all_embeddings = []

for path in board_image_paths:
    image = Image.open(path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
    embedding = outputs.pooler_output
    all_embeddings.append(embedding[0])

all_embeddings = torch.stack(all_embeddings)
print("All embeddings shape:", all_embeddings.shape)

vibe_vector = all_embeddings.mean(dim=0)
vibe_vector = vibe_vector / vibe_vector.norm()

print("Vibe vector shape:", vibe_vector.shape)
print("Vibe vector length (should be ~1.0):", vibe_vector.norm().item())

All embeddings shape: torch.Size([10, 512])
Vibe vector shape: torch.Size([512])
Vibe vector length (should be ~1.0): 1.0


## Save the vibe vector

This vibe vector is the output of this whole notebook, and later steps (item recommendations, zero-shot tags) will need it without recomputing everything above. We save it to `data/embeddings/` as a `.npy` file -- NumPy's binary format for arrays, which loads back instantly.

Note we first move the tensor off the GPU/MPS device and convert it to a plain NumPy array with `.cpu().numpy()`, since `.npy` files don't know about PyTorch tensors or devices.

In [5]:
import numpy as np

output_path = Path("../data/embeddings/board_vibe_vector.npy")
np.save(output_path, vibe_vector.cpu().numpy())

print(f"Saved vibe vector to {output_path}")

Saved vibe vector to ../data/embeddings/board_vibe_vector.npy
